# Greedy vs Joint pipeline extraction — throughput vs. p  (all-nodes)

각 모델 **total throughput** bar graph.
- 가장 왼쪽 = **greedy**(배포 해), 점선 세로선 뒤 = **joint** best @ p.
- **막대 = all-nodes 최적**: p개 파이프라인 구성에서 **각 그룹이 자기 노드(GPU)를 전부 사용** → 버려지는 GPU 없음.
- 메모리상 p개 모두 valid하게 구성 불가능한 p는 표시하지 않음 (Llama-3.1-70B 최대 **3**, Qwen3-32B 최대 **7**).
- `p=K`(greedy가 고른 개수) 막대 = **초록** = 전역 joint 최적(= greedy와 일치).
- 가로 점선 = greedy throughput 수준 (어떤 joint 막대도 넘지 못함 → greedy = 전역 최적). beam **k=3**.

데이터: `results/all_nodes_compare.json` (all_nodes_compare.py, top_k=3).

In [17]:
import os, sys

# 캐시된 config만 쓰려면(오프라인). `hf auth login` 돼 있으면 이 두 줄 제거 가능.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# optimizer 디렉토리(=joint_p_common.py 위치) 자동 탐색
d = os.getcwd()
while not os.path.exists(os.path.join(d, "joint_p_common.py")) and d != os.path.dirname(d):
    d = os.path.dirname(d)
OPT_DIR = d
os.chdir(OPT_DIR)
sys.path.insert(0, OPT_DIR)

import joint_p_common as J
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
print("optimizer dir:", OPT_DIR)

optimizer dir: /Users/swjeong/Desktop/ShuntServe/ArtifactEvaluation/ModelPlacement/optimizer/joint-comparison


In [ ]:
import json
MODELS = [("llama3-70b", "Llama-3.1-70B"), ("qwen3-32b", "Qwen3-32B")]

# all-nodes: 각 그룹이 자기 노드(GPU)를 전부 사용(버려지는 GPU 없음), beam k=3
AN = json.load(open("results/all_nodes_compare.json"))
data = {}
for ms, _title in MODELS:
    m = AN["models"][ms]
    per_p = {int(p): v for p, v in m["per_p"].items()}        # {p: total or None}
    data[ms] = {"per_p": per_p, "greedy": m["greedy"], "K": m["K"]}
    print(ms, "greedy=%.4f K=%d" % (m["greedy"], m["K"]),
          "| all-nodes best@p =", [None if per_p[p] is None else round(per_p[p], 3) for p in range(1, 10)])

In [ ]:
# ── 테스트하면서 바꿀 수 있는 값 ──────────────────────────────────────
XLABEL_FS = 24      # x축 라벨
XTICK_FS  = 20      # x축 눈금(greedy/1/2/...)
YLABEL_FS = 24      # y축 라벨
YTICK_FS  = 22      # y축 눈금
LEGEND_FS = 18      # 범례
FIG_W = 6.0         # 두 그림 공통 figure 가로 (동일 figsize)
FIG_H = 4.5         # 두 그림 공통 figure 세로
GAP   = 0.4         # greedy ↔ joint 사이 간격
PAD   = 0.7         # 좌우 대칭 여백 (막대 그룹을 가운데 정렬)
BAR_W = 0.7         # 막대 폭
YTOP  = 1.22        # y축 상한 = 최대치 * YTOP
HATCH_OPT = "//"    # optimal(greedy, p=K) 막대 동그라미 패턴
# ──────────────────────────────────────────────────────────────────────

os.makedirs("figures", exist_ok=True)
for ms, _t in MODELS:                                    # 모델별로 별도 figure (figsize 동일)
    dd = data[ms]; K = dd["K"]; g = dd["greedy"]
    valid_ps = [p for p in range(1, 10) if dd["per_p"][p] is not None]   # valid 구성 있는 p만
    heights = [g] + [dd["per_p"][p] for p in valid_ps]
    colors = ["#7f7f7f"] + ["#2ca02c" if p == K else "#4c78a8" for p in valid_ps]
    is_opt = [True] + [p == K for p in valid_ps]         # greedy + p=K = optimal
    xpos = [0] + [1 + GAP + k for k in range(len(valid_ps))]   # greedy=0, joint=1.4,2.4,...

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    bars = ax.bar(xpos, heights, width=BAR_W, color=colors, edgecolor="black", linewidth=0.6, zorder=3)
    for b, opt in zip(bars, is_opt):                     # optimal 막대에 동그라미 패턴
        if opt:
            b.set_hatch(HATCH_OPT)
    ax.axvline((xpos[0] + xpos[1]) / 2, color="black", ls="--", lw=1.3)   # greedy | joint 구분
    ax.axhline(g, color="#7f7f7f", ls=":", lw=1.0, zorder=1)              # greedy 수준
    ax.set_xticks(xpos)
    ax.set_xticklabels(["greedy"] + [str(p) for p in valid_ps], fontsize=XTICK_FS)
    ax.tick_params(axis="y", labelsize=YTICK_FS)
    ax.set_xlabel("# pipelines (P)", fontsize=XLABEL_FS)
    ax.set_ylabel("Total RPS (req/s)", fontsize=YLABEL_FS)
    ax.set_ylim(0, max(heights) * YTOP)                  # 상한 = 최대치 * 1.1
    ax.set_xlim(xpos[0] - PAD, xpos[-1] + PAD)           # 대칭 여백 → 그룹 가운데 정렬
    ax.legend(handles=[Patch(facecolor="white", edgecolor="black",
                             hatch=HATCH_OPT, label="optimal")],
              fontsize=LEGEND_FS, loc="upper right")
    plt.tight_layout()
    plt.savefig(f"figures/joint_vs_greedy_{ms}.pdf")     # bbox=tight 제거 → 두 그림 크기 동일
    plt.savefig(f"figures/joint_vs_greedy_{ms}.png", dpi=150)
    plt.show()
    print(f"saved -> figures/joint_vs_greedy_{ms}.{{pdf,png}}")